In [3]:
#!/usr/bin/env python3
# coding: utf-8
"""
upload_images_cloudinary_firestone.py

- Upload images in IMAGES_FOLDER to Cloudinary (folder BUCKET_FOLDER)
- Match grouped image base names to Firestore documents (collection FIRESTORE_COLLECTION)
- Update document's imageUrls (append or replace)
- Write unmatched groups to UNMATCHED_CSV for manual review
"""

import os
import re
import csv
import unicodedata
import cloudinary
import cloudinary.uploader
import firebase_admin
from firebase_admin import credentials, firestore
from rapidfuzz import fuzz

# ========== CONFIG ==========
CREDENTIALS_PATH = '../../private_key.json'  # caminho para sua service account JSON
IMAGES_FOLDER = r'C:\Users\Layanny\Documents\patrimonygo\src\assets\Patrimonios'  # ajuste conforme seu PC
CLOUD_NAME = 'dglfvvzg1'
CLOUD_API_KEY = '555873354438444'
CLOUD_API_SECRET = '_ELGOWW6c3l9-e9-94H_GrhoSfk'
BUCKET_FOLDER = 'patrimonios'  # pasta no Cloudinary
FIRESTORE_COLLECTION = 'patrimonios_santos'
FUZZY_THRESHOLD = 70  # ajuste; 70 é permissivo. Para maior segurança use 75-85
UPDATE_APPEND = True  # True -> concatena URLs ao campo imageUrls existente; False -> substitui
UNMATCHED_CSV = 'unmatched.csv'
# ============================ #

# Inicializa Cloudinary
cloudinary.config(
    cloud_name=CLOUD_NAME,
    api_key=CLOUD_API_KEY,
    api_secret=CLOUD_API_SECRET,
    secure=True
)

# Inicializa Firebase Admin
cred = credentials.Certificate(CREDENTIALS_PATH)
firebase_admin.initialize_app(cred)
db = firestore.client()

# ---------- Utilitários ----------

def extract_base(fname: str) -> str:
    """
    Extrai o 'nome base' do arquivo:
    - remove extensão
    - remove sufixo numérico do tipo " 01", "_01", "-01", "(01)" apenas se houver
    - mantém o resto do nome (ex: 'Necropole Ecumênica.jpg' -> 'Necropole Ecumênica')
    """
    base = os.path.splitext(fname)[0].strip()
    # remove apenas se houver sufixo numérico (um a três dígitos)
    base = re.sub(r'[\s\-_]*\(?\d{1,3}\)?$','', base).strip()
    return base

def normalize(text: str) -> str:
    """ Normaliza: remove acentos, pontuação (substitui por espaço), múltiplos espaços, lower """
    if text is None:
        return ''
    text = str(text)
    # normalize unicode (remove acentos)
    text = unicodedata.normalize('NFKD', text)
    text = ''.join(ch for ch in text if not unicodedata.combining(ch))
    # substituir pontuação por espaço
    text = re.sub(r'[^0-9A-Za-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip().lower()
    return text

def group_images(folder: str):
    """Agrupa arquivos de imagens pelo base name (extract_base)"""
    groups = {}
    if not os.path.isdir(folder):
        raise FileNotFoundError(f'Pasta de imagens não encontrada: {folder}')
    for fname in sorted(os.listdir(folder)):
        if not fname.lower().endswith(('.jpg', '.jpeg', '.png', '.webp', '.gif')):
            continue
        base = extract_base(fname)
        groups.setdefault(base, []).append(fname)
    return groups

def upload_image_to_cloudinary(local_path: str, desired_public_id: str = None) -> str:
    """
    Faz upload para Cloudinary e retorna secure_url. Usa desired_public_id (sem extensão).
    substitui espaços por underscore no public_id para URLs limpas.
    """
    public_id = None
    if desired_public_id:
        # sanitize public_id: remove acentos, pontuação e trocar espaços por _
        p = normalize(desired_public_id)
        p = p.replace(' ', '_')
        public_id = p
    try:
        res = cloudinary.uploader.upload(
            local_path,
            folder=BUCKET_FOLDER,
            public_id=public_id,
            overwrite=True,
            resource_type="image"
        )
        return res.get('secure_url')
    except Exception as e:
        print(f'  ❌ Erro ao enviar {local_path}: {e}')
        return None

def build_firestore_index():
    """
    Retorna:
      - index: dict mapping normalized_name -> list of (doc_id, original_name)
      - flat_names: list of tuples (doc_id, original_name, normalized_name)
    Tenta vários campos para o nome: name, nm_patrimonio, title, titulo
    """
    docs = list(db.collection(FIRESTORE_COLLECTION).stream())
    index = {}
    flat = []
    for d in docs:
        data = d.to_dict() or {}
        # tenta múltiplos campos possíveis para o nome
        name = data.get('name') or data.get('nm_patrimonio') or data.get('title') or data.get('titulo') or ''
        norm = normalize(name)
        index.setdefault(norm, []).append((d.id, name))
        flat.append((d.id, name, norm))
    return index, flat

def best_match_score(a_norm: str, flat_firestore):
    """
    Compara a_norm com cada normalized name no Firestore (flat_firestore).
    Usa token-based fuzzy (token_set, token_sort, partial) e retorna:
      (best_score (int), best_doc_id, best_orig_name, best_norm)
    """
    best_score = 0
    best_doc_id = None
    best_orig_name = None
    best_norm = None
    for doc_id, orig_name, norm_name in flat_firestore:
        s1 = fuzz.token_set_ratio(a_norm, norm_name)
        s2 = fuzz.token_sort_ratio(a_norm, norm_name)
        s3 = fuzz.partial_ratio(a_norm, norm_name)
        score = max(s1, s2, s3)
        if score > best_score:
            best_score = score
            best_doc_id = doc_id
            best_orig_name = orig_name
            best_norm = norm_name
    return int(best_score), best_doc_id, best_orig_name, best_norm

# ---------- Processo principal ----------

def process_all():
    groups = group_images(IMAGES_FOLDER)
    print(f'Grupos de imagens detectados: {len(groups)}')

    index, flat_firestore = build_firestore_index()
    print(f'Documentos indexados no Firestore: {len(flat_firestore)}')

    unmatched = []
    updated_count = 0

    for base_raw, files in groups.items():
        print(f'\n📍 Grupo: "{base_raw}"  ({len(files)} arquivos)')
        base_norm = normalize(base_raw)

        # 1) tentativa de match exato (normalizado)
        exact_docs = index.get(base_norm)
        matched_doc_id = None
        matched_name = None

        if exact_docs:
            # se houver mais de um, pega o primeiro (ou poderia escolher outro critério)
            matched_doc_id, matched_name = exact_docs[0]
            print(f'  ✅ Match exato encontrado: "{matched_name}" (doc {matched_doc_id})')
        else:
            # 2) fuzzy token-based
            score, doc_id, orig_name, norm = best_match_score(base_norm, flat_firestore)
            print(f'  🔎 Best fuzzy score: {score} (suggested: "{orig_name}" )' if orig_name else f'  🔎 Best fuzzy score: {score}')
            if score >= FUZZY_THRESHOLD:
                matched_doc_id = doc_id
                matched_name = orig_name
                print(f'  ✅ Aceito fuzzy match: "{matched_name}" (doc {matched_doc_id})')
            else:
                print(f'  ⚠️ Nenhum match confiável (melhor score {score}). Será marcado como unmatched.')
        
        # Faz upload das imagens e coleta URLs
        urls = []
        for fname in files:
            local_path = os.path.join(IMAGES_FOLDER, fname)
            # public_id opcional: usa o nome base + '_' + índice (para evitar colisão entre imagens do mesmo base)
            base_for_public = f"{base_raw}_{os.path.splitext(fname)[0].split()[-1]}" if ' ' in fname else os.path.splitext(fname)[0]
            # preferimos usar o basename sem extensão como public_id para legibilidade
            public_id = os.path.splitext(fname)[0]
            print(f'  -> Upload {fname} ...', end=' ')
            url = upload_image_to_cloudinary(local_path, desired_public_id=public_id)
            if url:
                urls.append(url)
                print('OK')
            else:
                print('FALHOU')

        # Atualiza Firestore se houver matching
        if matched_doc_id:
            doc_ref = db.collection(FIRESTORE_COLLECTION).document(matched_doc_id)
            try:
                if UPDATE_APPEND:
                    doc_snapshot = doc_ref.get()
                    existing = doc_snapshot.to_dict().get('imageUrls') or []
                    # evita duplicatas
                    new_urls = existing + [u for u in urls if u not in existing]
                    doc_ref.update({'imageUrls': new_urls})
                else:
                    doc_ref.update({'imageUrls': urls})
                print(f'  🔁 Firestore atualizado: {matched_doc_id} (nome registrado: "{matched_name}")')
                updated_count += 1
            except Exception as e:
                print(f'  ❌ Erro ao atualizar Firestore para {matched_doc_id}: {e}')
                unmatched.append((base_raw, ';'.join(files), 'update_error', str(e)))
        else:
            unmatched.append((base_raw, ';'.join(files), 'no_match', ''))
    
    # salvar unmatched para revisão
    if unmatched:
        with open(UNMATCHED_CSV, 'w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(['base_name', 'files', 'reason', 'detail'])
            writer.writerows(unmatched)
        print(f'\n⚠️ Há {len(unmatched)} grupos não correspondidos. Veja {UNMATCHED_CSV}')
    print(f'\n✔️ Finalizado. Documentos atualizados: {updated_count}')

if __name__ == '__main__':
    process_all()


Grupos de imagens detectados: 24
Documentos indexados no Firestore: 32

📍 Grupo: "ALFÂNDEGA DA RECEITA FEDERAL DO BRASIL EM SANTOS"  (3 arquivos)
  ✅ Match exato encontrado: "ALFÂNDEGA DA RECEITA FEDERAL DO BRASIL EM SANTOS" (doc kKEFGjaKqs57NVDbSl6I)
  -> Upload ALFÂNDEGA DA RECEITA FEDERAL DO BRASIL EM SANTOS 01.jpg ... OK
  -> Upload ALFÂNDEGA DA RECEITA FEDERAL DO BRASIL EM SANTOS 01.png ... OK
  -> Upload ALFÂNDEGA DA RECEITA FEDERAL DO BRASIL EM SANTOS 03.jpg ... OK
  🔁 Firestore atualizado: kKEFGjaKqs57NVDbSl6I (nome registrado: "ALFÂNDEGA DA RECEITA FEDERAL DO BRASIL EM SANTOS")

📍 Grupo: "BASÍLICA EMBARÉ"  (3 arquivos)
  ✅ Match exato encontrado: "BASÍLICA EMBARÉ" (doc mvAyx9YKG6wy2hIFzSHj)
  -> Upload BASÍLICA EMBARÉ 01.jpg ... OK
  -> Upload BASÍLICA EMBARÉ 02.jpg ... OK
  -> Upload BASÍLICA EMBARÉ 03.jpg ... OK
  🔁 Firestore atualizado: mvAyx9YKG6wy2hIFzSHj (nome registrado: "BASÍLICA EMBARÉ")

📍 Grupo: "BONDES - LINHA TURISTICA"  (2 arquivos)
  ✅ Match exato encontrado: "B